In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# --- Load Dataset ---
df = pd.read_csv('wesad_6features_balanced.csv')
features = ['bvp_mean', 'bvp_std', 'temp_mean', 'temp_std', 'acc_mag_mean', 'acc_mag_std']
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# --- Data Info & Missing Values ---
print(df.info())
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")
print(f"\nClass Distribution:\n{df['emotion'].value_counts()}")

In [ ]:
# --- Descriptive Statistics ---
df.describe().round(4)

In [ ]:
# --- Boxplots by Emotion ---
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for i, feat in enumerate(features):
    ax = axes[i // 3, i % 3]
    df.boxplot(column=feat, by='emotion', ax=ax, patch_artist=True,
               boxprops=dict(facecolor='lightblue'), medianprops=dict(color='red', linewidth=2))
    ax.set_title(feat, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
fig.suptitle('Feature Distributions by Emotion', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Correlation Heatmap ---
plt.figure(figsize=(8, 6))
sns.heatmap(df[features].corr(), annot=True, fmt='.3f', cmap='coolwarm', center=0, square=True)
plt.title('Feature Correlation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Encode Labels & Train-Test Split ---
le = LabelEncoder()
y = le.fit_transform(df['emotion'])
X = df[features].values

print(f"Label Mapping: {dict(zip(le.classes_, range(len(le.classes_))))}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")

In [ ]:
# --- Feature Scaling ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Scaled mean: {X_train_scaled.mean(axis=0).round(4)}")
print(f"Scaled std:  {X_train_scaled.std(axis=0).round(4)}")

In [ ]:
# --- Train & Compare Models ---
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='rbf', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
    'MLP': MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring='accuracy')
    model.fit(X_train_scaled, y_train)
    test_acc = accuracy_score(y_test, model.predict(X_test_scaled))
    print(f"{name:<22} CV: {cv_scores.mean():.4f}  Test: {test_acc:.4f}")

In [ ]:
# --- Evaluate Best Model (Random Forest) ---
best_model = models['Random Forest']
y_pred = best_model.predict(X_test_scaled)

print(classification_report(y_test, y_pred, target_names=le.classes_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=le.classes_).plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Counts')
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred, normalize='true'), display_labels=le.classes_).plot(ax=axes[1], cmap='Greens', colorbar=False, values_format='.2%')
axes[1].set_title('Normalized')
plt.tight_layout()
plt.show()

# Per-class accuracy
print('\nPER-CLASS ACCURACY:')
for i, cls in enumerate(le.classes_):
    mask = y_test == i
    cls_acc = accuracy_score(y_test[mask], y_pred[mask])
    print(f'  {cls}: {cls_acc:.4f} ({(y_test[mask] == y_pred[mask]).sum()}/{mask.sum()})')

In [ ]:
# --- Feature Importance ---
feat_imp = pd.Series(best_model.feature_importances_, index=features).sort_values(ascending=True)
feat_imp.plot(kind='barh', color='#4ECDC4', edgecolor='black', figsize=(8, 4))
plt.title('Feature Importance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# --- Predict New Input ---
def predict_emotion(bvp_mean, bvp_std, temp_mean, temp_std, acc_mag_mean, acc_mag_std):
    input_scaled = scaler.transform([[bvp_mean, bvp_std, temp_mean, temp_std, acc_mag_mean, acc_mag_std]])
    pred = best_model.predict(input_scaled)[0]
    probs = best_model.predict_proba(input_scaled)[0]
    emotion = le.inverse_transform([pred])[0]

    print(f"\nPredicted: {emotion}")
    for i, cls in enumerate(le.classes_):
        print(f"  {cls:<12} {probs[i]*100:6.2f}%")
    return emotion

# --- Test on sample values ---
predict_emotion(-0.229, 12.8382, 31.126, 0.008, 62.6925, 3.7878)

In [ ]:
# --- Save Best Model ---
import joblib

joblib.dump(best_model, 'best_model_random_forest.pkl')
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(le, 'label_encoder.pkl')

print('Saved:')
print('  best_model_random_forest.pkl')
print('  scaler.pkl')
print('  label_encoder.pkl')